# X2CT-GAN — Inter-Comparison Baseline for PPCNet
## Adapted from: Ying et al., "X2CT-GAN: Reconstructing CT from Biplanar X-Rays with Generative Adversarial Networks", CVPR 2019

### Adaptation Protocol (identical to 3D-ReVert inter-comparison)
| Aspect | What we keep from X2CT-GAN | What we keep from PPCNet v12.3 |
|---|---|---|
| **Encoder** | DenseNet-style 2D encoder (x2 views) with InstanceNorm | — |
| **Decoder** | 3D transposed-conv decoder with Connection-A/B/C fusion | — |
| **Discriminator** | 3D PatchGAN discriminator (LSGAN) | — |
| **Loss** | MSE reconstruction + LSGAN adversarial + projection loss | — |
| **LR schedule** | Adam lr=2e-4, beta1=0.5, beta2=0.99, linear decay after 50 ep | — |
| **Dataset** | — | Lumbar_Filtered_1037 (829/103/105) |
| **Coordinate system** | — | pts_to_local / local_to_world / compute_scale |
| **GT representation** | Volume (occupancy at 64 cubed) from GT point cloud | GT point cloud for eval |
| **Evaluation** | — | Chamfer, F@1/2/5 mm, HD95 (world-mm) |
| **N_POINTS** | — | 8192 |

### Key architectural differences from PPCNet v12.3
- **Output**: 3D occupancy volume (64 cubed) then extract point cloud at test time (vs direct point cloud)
- **Encoder**: DenseNet-style with InstanceNorm (vs ResNet-34 with BatchNorm)
- **Fusion**: Connection-C average fusion (vs BiplanarFusion concat+conv)
- **Loss**: MSE + adversarial + projection (vs Chamfer + gap + axial + extent + curvature + SW)
- **No projection matrices used**: X2CT-GAN does not use calibrated P matrices for query refinement
- **No iterative refinement**: single forward pass through encoder-decoder (vs 3-stage MLP refinement)


In [ ]:
"""
X2CT-GAN Inter-Comparison Baseline
====================================
Ying et al., CVPR 2019 — adapted for lumbar spine point cloud prediction.
Architecture: DenseNet encoder (2D) x 2 -> Connection-A/B/C -> 3D decoder -> 64^3 volume
Loss: MSE + LSGAN adversarial + projection loss (their original)
Evaluation: Chamfer, F@1/2/5mm, HD95 in world-mm (our standard metrics)
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import vtk
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.cuda.set_per_process_memory_fraction(min(50.0 / total_gb, 0.95), 0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# -- PATHS --
DATA_ROOT   = Path("/path/to/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("/path/to/inter_comparison_x2ctgan")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -- MODEL CONFIG (X2CT-GAN original settings) --
IMG_SIZE        = 128           # X2CT-GAN input resolution (resized from 512)
VOL_SIZE        = 64            # 3D output volume resolution
N_POINTS        = 8192          # point cloud size for evaluation (same as v12.3)
GROWTH_RATE     = 16            # dense block growth rate
DENSE_LAYERS    = 4             # layers per dense block

# -- TRAINING CONFIG (X2CT-GAN original settings) --
BATCH_SIZE      = 1             # X2CT-GAN: batch=1 (GPU memory for 3D volumes)
NUM_WORKERS     = 4
EPOCHS          = 100           # X2CT-GAN: 100 epochs
LR_G            = 2e-4          # X2CT-GAN: Adam lr=2e-4
LR_D            = 2e-4
BETA1           = 0.5           # X2CT-GAN: beta1=0.5
BETA2           = 0.99          # X2CT-GAN: beta2=0.99
LR_DECAY_START  = 50            # X2CT-GAN: linear decay after epoch 50
GRAD_CLIP       = 1.0
EARLY_STOP_PATIENCE = 60        # same patience as our framework

# -- LOSS WEIGHTS (X2CT-GAN original: lam1=0.1, lam2=10, lam3=10) --
LAMBDA_ADV      = 0.1           # adversarial (LSGAN)
LAMBDA_RECON    = 10.0          # MSE reconstruction
LAMBDA_PROJ     = 10.0          # projection loss (L1 on orthogonal projections)

MAX_EVAL        = 103
CKPT_PATH       = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT       = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG    = LOG_DIR  / "training_log.json"

print("=" * 65)
print("X2CT-GAN Inter-Comparison — Lumbar Spine Point Cloud")
print("=" * 65)
for k, v in dict(
    PAPER="Ying et al., CVPR 2019",
    INPUT=f"2 x {IMG_SIZE}x{IMG_SIZE} DRR (AP + LP)",
    OUTPUT=f"{VOL_SIZE}^3 occupancy volume -> {N_POINTS}-point cloud",
    LOSS=f"MSE(w={LAMBDA_RECON}) + LSGAN(w={LAMBDA_ADV}) + Proj(w={LAMBDA_PROJ})",
    OPTIMIZER=f"Adam (lr={LR_G}, beta1={BETA1}, beta2={BETA2})",
    LR_DECAY=f"Linear decay after epoch {LR_DECAY_START}",
    EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
    EVAL_METRICS="Chamfer, F@1/2/5mm, HD95 (world-mm)",
).items():
    print(f"  {k:<18} = {v}")


Device : cuda
GPU    : NVIDIA A100-SXM4-80GB MIG 1g.10gb
X2CT-GAN Inter-Comparison — Lumbar Spine Point Cloud
  PAPER              = Ying et al., CVPR 2019
  INPUT              = 2 x 128x128 DRR (AP + LP)
  OUTPUT             = 64^3 occupancy volume -> 8192-point cloud
  LOSS               = MSE(w=10.0) + LSGAN(w=0.1) + Proj(w=10.0)
  OPTIMIZER          = Adam (lr=0.0002, beta1=0.5, beta2=0.99)
  LR_DECAY           = Linear decay after epoch 50
  EPOCHS             = 100
  BATCH_SIZE         = 1
  EVAL_METRICS       = Chamfer, F@1/2/5mm, HD95 (world-mm)


In [2]:
# ==============================================================================
# DATA UTILITIES — same coordinate system as PPCNet v12.3
# ==============================================================================

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points: vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    """Load DRR and resize to X2CT-GAN input resolution (128x128)."""
    img = Image.open(path).convert("L")
    if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_projection_matrix(path):
    with open(path) as f: vals = [float(v) for v in f.read().split()]
    return np.array(vals, np.float32).reshape(3, 4)

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[None] + c[None]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))


def make_gt_volume(pts_local, vol_size=VOL_SIZE, dilation=1):
    """
    Convert GT point cloud (in local [-1,1]^3 coords) to a binary occupancy
    volume for X2CT-GAN training (their MSE reconstruction target).
    """
    p = np.clip((np.asarray(pts_local, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vol_size).astype(np.int32), 0, vol_size - 1)
    vol = np.zeros((vol_size,) * 3, np.float32)
    for dx in range(-dilation, dilation + 1):
        for dy in range(-dilation, dilation + 1):
            for dz in range(-dilation, dilation + 1):
                vol[np.clip(idx[:, 0] + dx, 0, vol_size - 1),
                    np.clip(idx[:, 1] + dy, 0, vol_size - 1),
                    np.clip(idx[:, 2] + dz, 0, vol_size - 1)] = 1.0
    return vol


def volume_to_points(vol, threshold=0.5, n_points=N_POINTS):
    """
    Extract point cloud from predicted occupancy volume.
    Returns points in local [-1,1]^3 coordinates.
    """
    mask = vol > threshold
    coords = np.argwhere(mask).astype(np.float32)  # (M, 3)
    if len(coords) == 0:
        # Fallback: if volume is empty, return random grid
        return np.random.uniform(-0.5, 0.5, (n_points, 3)).astype(np.float32)
    # Convert voxel indices to local coords: idx -> [-1, 1]
    local = (coords + 0.5) / vol.shape[0] * 2.0 - 1.0
    # Subsample or pad to n_points
    if len(local) >= n_points:
        idx = np.random.choice(len(local), n_points, replace=False)
        return local[idx]
    else:
        extra = n_points - len(local)
        pad_idx = np.random.choice(len(local), extra, replace=True)
        pad = local[pad_idx] + np.random.randn(extra, 3).astype(np.float32) * 0.01
        return np.concatenate([local, pad], 0)


class LumbarDataset(Dataset):
    """
    Dataset for X2CT-GAN inter-comparison.
    Returns: AP DRR, LP DRR (each 1x128x128), GT occupancy volume (64^3),
             GT point cloud (for evaluation), center/scale (for coordinate conversion).
    Note: X2CT-GAN does NOT use projection matrices — only the images.
    """
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids = ids; self.root = Path(root); self.aug = aug

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid

        # Load DRRs at 128x128 (X2CT-GAN resolution)
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")

        # Load GT point cloud
        gt = load_vtk_points(d / "gt_ppc.vtk")
        c  = gt.mean(0).astype(np.float32)
        s  = compute_scale(gt)
        gl = np.clip(pts_to_local(gt, c, s), -1, 1)

        # Create GT occupancy volume for MSE training
        gt_vol = make_gt_volume(gl)

        # Subsample GT for evaluation
        n = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))

        # Simple augmentation: horizontal flip only (X2CT-GAN uses minimal aug)
        if self.aug and np.random.rand() < 0.5:
            dap = dap[:, ::-1].copy()
            dlp = dlp[:, ::-1].copy()
            # Flip GT volume along X-axis
            gt_vol = gt_vol[::-1, :, :].copy()

        # Normalize to [0,1] — X2CT-GAN uses raw intensity, no ImageNet norm
        dap_t = torch.from_numpy(dap).unsqueeze(0).float()  # (1, 128, 128)
        dlp_t = torch.from_numpy(dlp).unsqueeze(0).float()

        return {
            "drr_ap":       dap_t,
            "drr_lp":       dlp_t,
            "gt_volume":    torch.from_numpy(gt_vol).float(),
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "gt_ppc_local": torch.from_numpy(gl[sel]).float(),
            "center":       torch.from_numpy(c).float(),
            "scale":        torch.from_numpy(s).float(),
            "patient_id":   pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")


Split: train=829  val=103  test=105


In [3]:
# ==============================================================================
# MODEL — X2CT-GAN (Ying et al., CVPR 2019)
# ==============================================================================
#
# Generator: 2 parallel dense-encoder -> Connection-A/B/C -> 3D decoder -> 64^3
# Discriminator: 3D PatchGAN with InstanceNorm (LSGAN)
#
# Architecture faithfully follows the paper:
#   - DenseNet-style 2D encoder per view (InstanceNorm, not BatchNorm)
#   - Connection-A at bottleneck (flatten -> FC -> reshape to 3D)
#   - Connection-B at skip connections (2D -> pseudo3D via expand + 3Dconv)
#   - Connection-C for fusion (permute to same coord frame -> average)
#   - 3D decoder with transposed convolutions + skip connections
# ==============================================================================


# -- Dense Encoder Building Blocks --

class DenseLayer2D(nn.Module):
    """Single layer in a dense block: IN -> ReLU -> Conv2D."""
    def __init__(self, in_ch, growth):
        super().__init__()
        self.layer = nn.Sequential(
            nn.InstanceNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, growth, 3, padding=1, bias=False))

    def forward(self, x):
        return torch.cat([x, self.layer(x)], 1)


class DenseBlock2D(nn.Module):
    """Dense block with n_layers dense layers."""
    def __init__(self, in_ch, growth=GROWTH_RATE, n_layers=DENSE_LAYERS):
        super().__init__()
        layers = []
        ch = in_ch
        for _ in range(n_layers):
            layers.append(DenseLayer2D(ch, growth))
            ch += growth
        self.block = nn.Sequential(*layers)
        self.out_ch = ch

    def forward(self, x):
        return self.block(x)


class DownBlock2D(nn.Module):
    """Downsample: stride-2 conv (as per X2CT-GAN paper)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Sequential(
            nn.InstanceNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False))

    def forward(self, x):
        return self.down(x)


class Compress2D(nn.Module):
    """Compress: 1x1 conv to halve channels (dense block output compression)."""
    def __init__(self, in_ch):
        super().__init__()
        self.compress = nn.Sequential(
            nn.InstanceNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, in_ch // 2, 1, bias=False))
        self.out_ch = in_ch // 2

    def forward(self, x):
        return self.compress(x)


# -- Connection Modules (paper Fig. 4) --

class ConnectionA(nn.Module):
    """
    Connection-A: 2D bottleneck -> 3D volume via flatten + FC + reshape.
    Bridges the 2D encoder final feature map to the 3D decoder first input.
    """
    def __init__(self, in_ch, spatial_2d, out_ch, spatial_3d):
        super().__init__()
        self.in_features = in_ch * spatial_2d * spatial_2d
        self.out_ch = out_ch
        self.spatial_3d = spatial_3d
        out_features = out_ch * spatial_3d * spatial_3d * spatial_3d
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.in_features, out_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5))

    def forward(self, x):
        B = x.shape[0]
        return self.fc(x).view(B, self.out_ch, self.spatial_3d,
                               self.spatial_3d, self.spatial_3d)


class ConnectionB(nn.Module):
    """
    Connection-B: 2D skip feature -> pseudo-3D via expand along depth + 3D conv.
    Used at each skip connection level between encoder and decoder.
    """
    def __init__(self, in_ch_2d, out_ch_3d, depth):
        super().__init__()
        self.depth = depth
        # Step 1: match 2D channel count to decoder side
        self.conv2d = nn.Sequential(
            nn.Conv2d(in_ch_2d, out_ch_3d, 1, bias=False),
            nn.InstanceNorm2d(out_ch_3d),
            nn.ReLU(inplace=True))
        # Step 2: refine the expanded pseudo-3D volume
        self.conv3d = nn.Sequential(
            nn.Conv3d(out_ch_3d, out_ch_3d, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch_3d),
            nn.ReLU(inplace=True))

    def forward(self, x_2d):
        x = self.conv2d(x_2d)                                    # (B, C, H, W)
        x = x.unsqueeze(2).expand(-1, -1, self.depth, -1, -1)    # (B, C, D, H, W)
        x = x.contiguous()
        return self.conv3d(x)


# -- 3D Decoder Building Blocks --

class UpBlock3D(nn.Module):
    """3D transposed convolution for upsampling."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, x):
        return self.up(x)


class Basic3D(nn.Module):
    """Basic 3D conv block: Conv3D -> IN -> ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


# -- Single-View Encoder --

class X2CTEncoder(nn.Module):
    """
    DenseNet-style 2D encoder for one view.
    Input: (B, 1, 128, 128)
    Produces feature maps at 4 spatial resolutions for skip connections.
    """
    def __init__(self):
        super().__init__()
        # Initial conv: 1 -> 32, 128->64
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(32),
            nn.ReLU(inplace=True))

        # Stage 1: 64x64 -> 32x32
        self.dense1 = DenseBlock2D(32);  c1 = self.dense1.out_ch
        self.comp1  = Compress2D(c1);    c1c = self.comp1.out_ch
        self.down1  = DownBlock2D(c1c, 64)

        # Stage 2: 32x32 -> 16x16
        self.dense2 = DenseBlock2D(64);  c2 = self.dense2.out_ch
        self.comp2  = Compress2D(c2);    c2c = self.comp2.out_ch
        self.down2  = DownBlock2D(c2c, 128)

        # Stage 3: 16x16 -> 8x8
        self.dense3 = DenseBlock2D(128); c3 = self.dense3.out_ch
        self.comp3  = Compress2D(c3);    c3c = self.comp3.out_ch
        self.down3  = DownBlock2D(c3c, 256)

        # Stage 4: 8x8 -> 4x4
        self.dense4 = DenseBlock2D(256); c4 = self.dense4.out_ch
        self.comp4  = Compress2D(c4);    c4c = self.comp4.out_ch
        self.down4  = DownBlock2D(c4c, 512)

        # Bottleneck at 4x4
        self.dense5 = DenseBlock2D(512); c5 = self.dense5.out_ch
        self.comp5  = Compress2D(c5);    self.bottleneck_ch = self.comp5.out_ch

        # Store skip channel counts for Connection-B
        self.skip_channels = [c1c, c2c, c3c, c4c]  # at 64, 32, 16, 8

    def forward(self, x):
        x0 = self.init_conv(x)                            # (B, 32, 64, 64)
        d1 = self.comp1(self.dense1(x0))                   # skip1 @ 64x64
        x1 = self.down1(d1)                                # (B, 64, 32, 32)
        d2 = self.comp2(self.dense2(x1))                   # skip2 @ 32x32
        x2 = self.down2(d2)                                # (B, 128, 16, 16)
        d3 = self.comp3(self.dense3(x2))                   # skip3 @ 16x16
        x3 = self.down3(d3)                                # (B, 256, 8, 8)
        d4 = self.comp4(self.dense4(x3))                   # skip4 @ 8x8
        x4 = self.down4(d4)                                # (B, 512, 4, 4)
        bn = self.comp5(self.dense5(x4))                   # bottleneck @ 4x4
        return bn, [d1, d2, d3, d4]


# -- Single-View Decoder --

class X2CTSingleViewDecoder(nn.Module):
    """
    3D decoder for one view encoder path.
    Takes bottleneck (via Connection-A) and skip features (via Connection-B).
    Output: (B, 32, 64, 64, 64)
    """
    def __init__(self, bottleneck_ch, skip_channels):
        super().__init__()
        # Connection-A: 2D bottleneck (4x4) -> 3D (4x4x4)
        self.conn_a = ConnectionA(bottleneck_ch, 4, 512, 4)

        # Connection-Bs: 2D skip -> 3D at each level
        self.conn_b4 = ConnectionB(skip_channels[3], 256, 8)   # skip4 @ 8
        self.conn_b3 = ConnectionB(skip_channels[2], 128, 16)  # skip3 @ 16
        self.conn_b2 = ConnectionB(skip_channels[1], 64, 32)   # skip2 @ 32
        self.conn_b1 = ConnectionB(skip_channels[0], 32, 64)   # skip1 @ 64

        # 3D decoder path: 4^3 -> 8^3 -> 16^3 -> 32^3 -> 64^3
        self.up4  = UpBlock3D(512, 256);   self.ref4 = Basic3D(256 + 256, 256)
        self.up3  = UpBlock3D(256, 128);   self.ref3 = Basic3D(128 + 128, 128)
        self.up2  = UpBlock3D(128, 64);    self.ref2 = Basic3D(64 + 64, 64)
        self.up1  = UpBlock3D(64, 32);     self.ref1 = Basic3D(32 + 32, 32)

    def forward(self, bottleneck_2d, skips):
        x = self.conn_a(bottleneck_2d)                                  # (B, 512, 4, 4, 4)
        x = self.ref4(torch.cat([self.up4(x), self.conn_b4(skips[3])], 1))  # -> 8^3
        x = self.ref3(torch.cat([self.up3(x), self.conn_b3(skips[2])], 1))  # -> 16^3
        x = self.ref2(torch.cat([self.up2(x), self.conn_b2(skips[1])], 1))  # -> 32^3
        x = self.ref1(torch.cat([self.up1(x), self.conn_b1(skips[0])], 1))  # -> 64^3
        return x  # (B, 32, 64, 64, 64)


# -- Full Generator --

class X2CTGenerator(nn.Module):
    """
    X2CT-GAN Generator — full biplanar architecture.
    Two parallel encoder-decoders (AP + LP) fused via Connection-C (average).
    """
    def __init__(self):
        super().__init__()
        self.encoder_ap = X2CTEncoder()
        self.encoder_lp = X2CTEncoder()
        bn_ch = self.encoder_ap.bottleneck_ch
        sk_ch = self.encoder_ap.skip_channels
        self.decoder_ap = X2CTSingleViewDecoder(bn_ch, sk_ch)
        self.decoder_lp = X2CTSingleViewDecoder(bn_ch, sk_ch)

        # Connection-C fusion + final refinement
        self.fusion_refine = nn.Sequential(
            Basic3D(32, 32),
            Basic3D(32, 16),
            nn.Conv3d(16, 1, 1))

    def forward(self, x_ap, x_lp):
        bn_ap, sk_ap = self.encoder_ap(x_ap)
        bn_lp, sk_lp = self.encoder_lp(x_lp)
        vol_ap = self.decoder_ap(bn_ap, sk_ap)
        vol_lp = self.decoder_lp(bn_lp, sk_lp)
        # Connection-C: permute LP volume to match AP frame, then average
        vol_lp_perm = vol_lp.permute(0, 1, 2, 4, 3).contiguous()
        fused = 0.5 * (vol_ap + vol_lp_perm)
        out = self.fusion_refine(fused)
        return torch.sigmoid(out.squeeze(1))  # (B, 64, 64, 64) in [0, 1]


# -- Discriminator --

class X2CTDiscriminator(nn.Module):
    """
    3D PatchGAN Discriminator (paper section 4.2).
    Conditional on input X-rays (expanded from 2D to 3D).
    """
    def __init__(self):
        super().__init__()
        # 1 volume channel + 2 conditional X-ray channels = 3
        self.net = nn.Sequential(
            nn.Conv3d(3, 64, 4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(64, 128, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm3d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(128, 256, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm3d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(256, 512, 4, stride=1, padding=1, bias=False),
            nn.InstanceNorm3d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(512, 1, 4, stride=1, padding=1))

    def forward(self, vol, x_ap, x_lp):
        B = vol.shape[0]
        V = vol.shape[1]
        # Expand 2D X-rays to 3D
        xap_3d = F.interpolate(x_ap, size=V, mode='bilinear', align_corners=False)
        xlp_3d = F.interpolate(x_lp, size=V, mode='bilinear', align_corners=False)
        xap_3d = xap_3d.unsqueeze(2).expand(B, 1, V, V, V)
        xlp_3d = xlp_3d.unsqueeze(3).expand(B, 1, V, V, V)
        x = torch.cat([vol.unsqueeze(1), xap_3d, xlp_3d], 1)
        return self.net(x)


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_g = X2CTGenerator();  print(f"Generator params:     {count_params(_g)/1e6:.1f} M"); del _g
_d = X2CTDiscriminator(); print(f"Discriminator params: {count_params(_d)/1e6:.1f} M"); del _d


Generator params:     344.1 M
Discriminator params: 11.1 M


In [4]:
# ==============================================================================
# LOSS FUNCTIONS — X2CT-GAN original (Ying et al., CVPR 2019)
# ==============================================================================
# 1. LSGAN adversarial loss (Eq. 2 in paper)
# 2. MSE reconstruction loss (Eq. 3 in paper)
# 3. Projection loss — L1 on orthogonal projections (Eq. 4 in paper)
# Total: G* = argmin [lam1*LSGAN(G) + lam2*MSE + lam3*Projection]


def lsgan_loss_D(D_real, D_fake):
    """LSGAN discriminator loss (Eq. 2 — discriminator part)."""
    return 0.5 * (torch.mean((D_real - 1.0) ** 2) + torch.mean(D_fake ** 2))


def lsgan_loss_G(D_fake):
    """LSGAN generator loss (Eq. 2 — generator part)."""
    return 0.5 * torch.mean((D_fake - 1.0) ** 2)


def mse_reconstruction_loss(pred_vol, gt_vol):
    """MSE reconstruction loss (Eq. 3 in paper)."""
    return F.mse_loss(pred_vol, gt_vol)


def projection_loss(pred_vol, gt_vol):
    """
    Projection loss (Eq. 4 in paper).
    L1 distance between orthogonal projections (sum along each axis)
    of predicted and GT volumes, averaged over 3 planes.
    Axial = sum along Z, Coronal = sum along Y, Sagittal = sum along X.
    """
    # Axial projection (sum along dim 1 = Z-axis)
    proj_ax_pred = pred_vol.sum(dim=1)
    proj_ax_gt   = gt_vol.sum(dim=1)

    # Coronal projection (sum along dim 2 = Y-axis)
    proj_co_pred = pred_vol.sum(dim=2)
    proj_co_gt   = gt_vol.sum(dim=2)

    # Sagittal projection (sum along dim 3 = X-axis)
    proj_sa_pred = pred_vol.sum(dim=3)
    proj_sa_gt   = gt_vol.sum(dim=3)

    loss = (F.l1_loss(proj_ax_pred, proj_ax_gt) +
            F.l1_loss(proj_co_pred, proj_co_gt) +
            F.l1_loss(proj_sa_pred, proj_sa_gt)) / 3.0
    return loss


print("X2CT-GAN loss functions defined:")
print(f"  1. LSGAN adversarial    (w = {LAMBDA_ADV})")
print(f"  2. MSE reconstruction   (w = {LAMBDA_RECON})")
print(f"  3. Projection (L1, 3pl) (w = {LAMBDA_PROJ})")


X2CT-GAN loss functions defined:
  1. LSGAN adversarial    (w = 0.1)
  2. MSE reconstruction   (w = 10.0)
  3. Projection (L1, 3pl) (w = 10.0)


In [5]:
# ==============================================================================
# TRAINING LOOP — X2CT-GAN: alternating G/D, Adam, linear LR decay
# ==============================================================================

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}

def chamfer_np(pred, gt):
    """Evaluation metrics — same as PPCNet v12.3."""
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th): p, r = (dp<=th).mean(), (dg<=th).mean(); return 2*p*r/(p+r) if p+r>0 else 0.
    return {"chamfer_mm": float(0.5*(dp.mean()+dg.mean())),
            "fscore_1mm": float(fs(1)), "fscore_2mm": float(fs(2)),
            "fscore_5mm": float(fs(5)),
            "hausdorff_95": float(max(np.percentile(dp,95), np.percentile(dg,95)))}


# -- Build models --
gen  = X2CTGenerator().to(device)
disc = X2CTDiscriminator().to(device)
print(f"Generator:     {count_params(gen)/1e6:.1f} M params")
print(f"Discriminator: {count_params(disc)/1e6:.1f} M params")

# X2CT-GAN optimizers: Adam, lr=2e-4, beta1=0.5, beta2=0.99
opt_G = torch.optim.Adam(gen.parameters(),  lr=LR_G, betas=(BETA1, BETA2))
opt_D = torch.optim.Adam(disc.parameters(), lr=LR_D, betas=(BETA1, BETA2))

train_ds     = LumbarDataset(train_ids, aug=True)
val_ds       = LumbarDataset(val_ids,   aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

print(f"Train : {len(train_ds)} samples -> {len(train_loader)} batches/ep")
print(f"Val   : {len(val_ds)} samples -> {len(val_loader)} batches/ep")


def lr_lambda(epoch):
    """X2CT-GAN: constant LR for first 50 epochs, then linear decay to 0."""
    if epoch < LR_DECAY_START:
        return 1.0
    return max(0.0, 1.0 - (epoch - LR_DECAY_START) / (EPOCHS - LR_DECAY_START))

sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)


def save_ckpt(path, ep, best, hist):
    tmp = path.with_suffix(".tmp")
    torch.save({"gen": gen.state_dict(), "disc": disc.state_dict(),
                "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict(),
                "sched_G": sched_G.state_dict(), "sched_D": sched_D.state_dict(),
                "epoch": ep, "best_chamfer": best, "history": hist,
                "config": {"method": "X2CT-GAN", "paper": "Ying2019"}}, tmp)
    tmp.replace(path)
    print(f"  Saved -> {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    gen.load_state_dict(st["gen"]); disc.load_state_dict(st["disc"])
    opt_G.load_state_dict(st["opt_G"]); opt_D.load_state_dict(st["opt_D"])
    try: sched_G.load_state_dict(st["sched_G"]); sched_D.load_state_dict(st["sched_D"])
    except: print("  (scheduler state mismatch — reinitializing)")
    ep = st["epoch"] + 1; bc = st.get("best_chamfer", float("inf")); h = st.get("history", [])
    print(f"  Resumed from ep {ep} | best={bc:.3f} mm")
    return ep, bc, h


start_epoch, best_chamfer, history, no_improve = 0, float("inf"), [], 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint — starting fresh.")

print(f"{'='*65}")
print(f"X2CT-GAN training from ep {start_epoch+1}/{EPOCHS}")
print(f"Loss: w_adv={LAMBDA_ADV}  w_recon={LAMBDA_RECON}  w_proj={LAMBDA_PROJ}")
print(f"LR={LR_G}, linear decay after ep {LR_DECAY_START}")
print(f"{'='*65}")


for epoch in range(start_epoch, EPOCHS):
    gen.train(); disc.train()
    t0 = time.time()
    acc_g, acc_d, nb = {}, {}, 0

    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=130)

    for bi, batch in pbar:
        batch = to_dev(batch)
        x_ap = batch["drr_ap"]
        x_lp = batch["drr_lp"]
        gt_vol = batch["gt_volume"]

        # -- Train Discriminator --
        opt_D.zero_grad(set_to_none=True)
        with torch.no_grad():
            fake_vol = gen(x_ap, x_lp)

        D_real = disc(gt_vol, x_ap, x_lp)
        D_fake = disc(fake_vol.detach(), x_ap, x_lp)
        loss_D = LAMBDA_ADV * lsgan_loss_D(D_real, D_fake)

        if torch.isfinite(loss_D):
            loss_D.backward()
            torch.nn.utils.clip_grad_norm_(disc.parameters(), GRAD_CLIP)
            opt_D.step()

        # -- Train Generator --
        opt_G.zero_grad(set_to_none=True)
        fake_vol = gen(x_ap, x_lp)

        D_fake_for_G = disc(fake_vol, x_ap, x_lp)
        loss_adv   = LAMBDA_ADV * lsgan_loss_G(D_fake_for_G)
        loss_recon = LAMBDA_RECON * mse_reconstruction_loss(fake_vol, gt_vol)
        loss_proj  = LAMBDA_PROJ * projection_loss(fake_vol, gt_vol)
        loss_G = loss_adv + loss_recon + loss_proj

        if torch.isfinite(loss_G):
            loss_G.backward()
            torch.nn.utils.clip_grad_norm_(gen.parameters(), GRAD_CLIP)
            opt_G.step()

        acc_g["adv"]   = acc_g.get("adv", 0)   + float(loss_adv.detach())
        acc_g["recon"] = acc_g.get("recon", 0)  + float(loss_recon.detach())
        acc_g["proj"]  = acc_g.get("proj", 0)   + float(loss_proj.detach())
        acc_g["total"] = acc_g.get("total", 0)  + float(loss_G.detach())
        acc_d["D"]     = acc_d.get("D", 0)      + float(loss_D.detach())
        nb += 1

        if bi % 50 == 0 or bi == len(train_loader):
            g = lambda k: acc_g.get(k, 0) / max(1, nb)
            pbar.set_postfix_str(
                f"G={g('total'):.3f} recon={g('recon'):.3f} "
                f"adv={g('adv'):.4f} proj={g('proj'):.3f} "
                f"D={acc_d.get('D',0)/max(1,nb):.4f}")

    sched_G.step(); sched_D.step()
    elapsed = time.time() - t0

    # -- Validation --
    gen.eval(); metrics = []; nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            pred_vol = gen(batch["drr_ap"], batch["drr_lp"])

            pred_vol_np = pred_vol.cpu().numpy()
            gw_np = batch["gt_ppc_world"].cpu().numpy()
            centers = batch["center"].cpu().numpy()
            scales  = batch["scale"].cpu().numpy()

            for b in range(pred_vol_np.shape[0]):
                if nd >= MAX_EVAL: break
                pred_local = volume_to_points(pred_vol_np[b], threshold=0.5,
                                              n_points=N_POINTS)
                pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
                metrics.append(chamfer_np(pred_world, gw_np[b]))
                nd += 1

    if metrics:
        mc = np.mean([m["chamfer_mm"]   for m in metrics])
        f2 = np.mean([m["fscore_2mm"]   for m in metrics])
        hd = np.mean([m["hausdorff_95"] for m in metrics])
        g  = lambda k: acc_g.get(k, 0) / max(1, nb)
        lr_now = sched_G.get_last_lr()[0]

        print(f"\n[Ep {epoch+1}] {elapsed:.0f}s  lr={lr_now:.2e}  "
              f"Val: Chamfer={mc:.3f}mm  F@2mm={f2:.3f}  HD95={hd:.3f}mm")
        print(f"  G: recon={g('recon'):.3f} adv={g('adv'):.4f} proj={g('proj'):.3f} "
              f" D={acc_d.get('D',0)/max(1,nb):.4f}")

        rec = {"epoch": epoch+1, "chamfer_mm": mc, "f2": f2, "hd95": hd,
               "train_recon": g("recon"), "train_adv": g("adv"),
               "train_proj": g("proj"), "lr": lr_now}
        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)

        if mc < best_chamfer:
            best_chamfer = mc; no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stop: {no_improve} epochs without improvement"); break

print(f"Done. Best val Chamfer: {best_chamfer:.3f} mm")


Generator:     344.1 M params
Discriminator: 11.1 M params
Train : 829 samples -> 829 batches/ep
Val   : 103 samples -> 103 batches/ep
No checkpoint — starting fresh.
X2CT-GAN training from ep 1/100
Loss: w_adv=0.1  w_recon=10.0  w_proj=10.0
LR=0.0002, linear decay after ep 50


Ep   1/100:   0%|                                                                                         | 0/829 [00:00<?, ?it/s]/tmp/ipykernel_4346/2411567721.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_4346/2411567721.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_4346/2411567721.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_4346/2411567721.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR i


[Ep 1] 346s  lr=2.00e-04  Val: Chamfer=8.632mm  F@2mm=0.275  HD95=41.039mm
  G: recon=1.319 adv=0.0539 proj=167.037  D=0.0100
  Saved -> latest_checkpoint.pth  (ep 1)
  Saved -> best_checkpoint.pth  (ep 1)
  New best: 8.632 mm


Ep   2/100: 100%|█████████████████████████| 829/829 [05:30<00:00,  2.50it/s, G=82.921 recon=0.583 adv=0.0508 proj=82.287 D=0.0022]



[Ep 2] 331s  lr=2.00e-04  Val: Chamfer=2.854mm  F@2mm=0.450  HD95=9.758mm
  G: recon=0.583 adv=0.0508 proj=82.287  D=0.0022
  Saved -> latest_checkpoint.pth  (ep 2)
  Saved -> best_checkpoint.pth  (ep 2)
  New best: 2.854 mm


...


[Ep 98] 331s  lr=8.00e-06  Val: Chamfer=2.330mm  F@2mm=0.454  HD95=5.812mm
  G: recon=0.051 adv=0.0500 proj=2.433  D=0.0000
  Saved -> latest_checkpoint.pth  (ep 98)


Ep  99/100: 100%|███████████████████████████| 829/829 [05:30<00:00,  2.51it/s, G=2.507 recon=0.050 adv=0.0500 proj=2.407 D=0.0000]



[Ep 99] 331s  lr=4.00e-06  Val: Chamfer=2.330mm  F@2mm=0.455  HD95=5.821mm
  G: recon=0.050 adv=0.0500 proj=2.407  D=0.0000
  Saved -> latest_checkpoint.pth  (ep 99)


Ep 100/100: 100%|███████████████████████████| 829/829 [05:30<00:00,  2.51it/s, G=2.504 recon=0.050 adv=0.0500 proj=2.403 D=0.0000]



[Ep 100] 331s  lr=0.00e+00  Val: Chamfer=2.331mm  F@2mm=0.454  HD95=5.825mm
  G: recon=0.050 adv=0.0500 proj=2.403  D=0.0000
  Saved -> latest_checkpoint.pth  (ep 100)
Done. Best val Chamfer: 2.324 mm


In [ ]:
# ==============================================================================
# TEST EVALUATION — X2CT-GAN: volume -> point cloud -> metrics
# ==============================================================================
import gc, torch

# ── Free ALL GPU memory left over from training ───────────────────────────────
try: del opt_g, opt_d, scheduler_g, scheduler_d
except NameError: pass
try: del disc
except NameError: pass
try: del gen                          # critical — free training gen from GPU
except NameError: pass

torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info(0)[0] / 1e9
    total = torch.cuda.mem_get_info(0)[1] / 1e9
    print(f"GPU memory free before test load: {free:.2f} / {total:.2f} GB")

# ── Rebuild generator fresh, load checkpoint safely via CPU ───────────────────
print("Test evaluation...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)  # CPU first
    gen = X2CTGenerator().cpu()
    gen.load_state_dict(st["gen"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val Chamfer {st['best_chamfer']:.3f} mm)")
    del st                            # free checkpoint dict before moving to GPU
    gc.collect()
    gen = gen.to(device)
else:
    print("  WARNING: No best checkpoint — using current model state.")
    gen = X2CTGenerator().to(device)

if torch.cuda.is_available():
    free = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"GPU memory free after model load:  {free:.2f} / {total:.2f} GB")

gen.eval()

# ── Test DataLoader ───────────────────────────────────────────────────────────
test_ds     = LumbarDataset(test_ids, aug=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn)

# ── Inference loop ────────────────────────────────────────────────────────────
all_metrics = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Test', ncols=100):
        batch = to_dev(batch)
        B = batch["drr_ap"].shape[0]
        pred_vol    = gen(batch["drr_ap"], batch["drr_lp"])
        pred_vol_np = pred_vol.cpu().numpy()
        gw_np       = batch["gt_ppc_world"].cpu().numpy()
        centers     = batch["center"].cpu().numpy()
        scales      = batch["scale"].cpu().numpy()
        pids        = batch["patient_id"]
        for b in range(B):
            pred_local = volume_to_points(pred_vol_np[b], threshold=0.5,
                                          n_points=N_POINTS)
            pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
            m = chamfer_np(pred_world, gw_np[b])
            m["patient_id"] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pred_world, RESULTS_DIR / f"{pids[b]}_pred.vtk")

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"TEST RESULTS — X2CT-GAN ({len(all_metrics)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm",  "Chamfer mm  "),
                 ("fscore_1mm",  "F-score @1mm"),
                 ("fscore_2mm",  "F-score @2mm"),
                 ("fscore_5mm",  "F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_metrics]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

# ── Save CSV ──────────────────────────────────────────────────────────────────
csv_path = RESULTS_DIR / "test_results_x2ctgan.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["patient_id", "chamfer_mm",
                                      "fscore_1mm", "fscore_2mm",
                                      "fscore_5mm", "hausdorff_95"])
    w.writeheader()
    w.writerows(all_metrics)
print(f"\nCSV -> {csv_path}")

In [ ]:
# ==============================================================================
# INTER-COMPARISON TABLE — read all CSVs and print side-by-side
# ==============================================================================

import pandas as pd

results = {}

# X2CT-GAN results
x2ct_csv = RESULTS_DIR / "test_results_x2ctgan.csv"
if x2ct_csv.exists():
    df = pd.read_csv(x2ct_csv)
    results["X2CT-GAN"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("X2CT-GAN results loaded.")

# Try to load PPCNet v12.3 results if available
v123_csv = Path("/path/to/inter_comparison_ppcnet_v12_3/results/test_results_v12_3_tta.csv")
if v123_csv.exists():
    df = pd.read_csv(v123_csv)
    results["PPCNet v12.3 (Ours)"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("PPCNet v12.3 results loaded.")

# Try to load 3D-ReVert results if available
revert_csv = Path("/path/to/inter_comparison_3drevert/results/test_results_3drevert.csv")
if revert_csv.exists():
    df = pd.read_csv(revert_csv)
    results["3D-ReVert"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("3D-ReVert results loaded.")

if results:
    print(f"\n{'='*80}")
    print("INTER-COMPARISON TABLE")
    print(f"{'='*80}")
    header = f"{'Method':<25} {'Chamfer (mm)':<20} {'F@1mm':<8} {'F@2mm':<8} {'F@5mm':<8} {'HD95 (mm)':<20}"
    print(header)
    print("-" * len(header))
    for method, vals in results.items():
        print(f"{method:<25} {vals['Chamfer']:<20} {vals['F@1mm']:<8} "
              f"{vals['F@2mm']:<8} {vals['F@5mm']:<8} {vals['HD95']:<20}")
else:
    print("No results CSVs found yet — run training first.")


X2CT-GAN results loaded.
PPCNet v12.3 results loaded.

INTER-COMPARISON TABLE
Method                    Chamfer (mm)         F@1mm    F@2mm    F@5mm    HD95 (mm)           
----------------------------------------------------------------------------------------------
X2CT-GAN                  2.489 +/- 0.928      0.079    0.444    0.933    6.624 +/- 4.673     
PPCNet v12.3 (Ours)       1.981 +/- 1.065      0.155    0.646    0.973    4.525 +/- 5.407     
